In [1]:
# -----------------------------
# 1. Import libraries
# -----------------------------
import pandas as pd
import yfinance as yf
import requests
from io import StringIO
import plotly.express as px
import ta
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# 2. Load S&P500 company list
# -----------------------------
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)

# Read the first table from Wikipedia
sp500 = pd.read_html(StringIO(response.text))[0]

# Keep only relevant columns
sp500 = sp500[["Symbol","GICS Sector","GICS Sub-Industry"]]

# Convert Wikipedia tickers to Yahoo Finance format (BRK.B → BRK-B)
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)

# Rename column to match price data
sp500 = sp500.rename(columns={"Symbol":"Ticker"})

tickers = sp500["Ticker"].tolist()
print(f"Total S&P500 companies: {len(tickers)}")

# -----------------------------
# 3. Download price data
# -----------------------------
prices = yf.download(
    tickers,
    start="2015-01-01",
    auto_adjust=True,
    progress=False
)

# Keep only closing prices
prices = prices["Close"]

# -----------------------------
# 4. Convert wide → long format
# -----------------------------
df_long = prices.stack().reset_index()
df_long.columns = ["Date","Ticker","Adj Close"]

# -----------------------------
# 5. Merge sector info
# -----------------------------
df_long = df_long.merge(
    sp500,
    on="Ticker",
    how="left"
)

df_long = df_long.rename(columns={
    "GICS Sector":"Sector",
    "GICS Sub-Industry":"SubIndustry"
})

# -----------------------------
# 6. Sort and reset index
# -----------------------------
df_long = df_long.sort_values(["Ticker","Date"]).reset_index(drop=True)

# -----------------------------
# 7. Compute daily returns
# -----------------------------
df_long["Return"] = df_long.groupby("Ticker")["Adj Close"].pct_change()

# -----------------------------
# 8. Save dataset (optional)
# -----------------------------
# df_long.to_parquet("sp500_price_panel.parquet")  # save for faster reload

# -----------------------------
# 9. Inspect dataset
# -----------------------------
print(df_long.shape)
df_long.head()

Total S&P500 companies: 503
(1378080, 6)


,Date,Ticker,Adj Close,Sector,SubIndustry,Return
0,2015-01-02,A,36.970371,Health Care,Life Sciences Tools & Services,NaN
1,2015-01-05,A,36.277626,Health Care,Life Sciences Tools & Services,-0.018738
2,2015-01-06,A,35.712509,Health Care,Life Sciences Tools & Services,-0.015578
3,2015-01-07,A,36.186481,Health Care,Life Sciences Tools & Services,0.013272
4,2015-01-08,A,37.271168,Health Care,Life Sciences Tools & Services,0.029975


In [2]:
# -----------------------------
# 1. Pre-compute RSI
# -----------------------------
df_long['RSI'] = df_long.groupby('Ticker')['Adj Close'].transform(
    lambda x: ta.momentum.RSIIndicator(x, window=14).rsi()
)

df_rsi_full = df_long.groupby('Ticker').tail(1).copy()

# -----------------------------
# 2. Widgets
# -----------------------------
tick_rsi = widgets.Text(description='Ticker:', placeholder='AAPL', continuous_update=False)

sec_rsi = widgets.Dropdown(
    description='Sector:',
    options=['All'] + sorted(df_rsi_full['Sector'].dropna().unique())
)

sub_rsi = widgets.Dropdown(description='Sub-Industry:', options=['All'])

out_rsi = widgets.Output()

# -----------------------------
# 3. Update Subindustries
# -----------------------------
def update_subs_rsi(change):

    if sec_rsi.value == 'All':
        sub_rsi.options = ['All']
    else:
        subs = sorted(
            df_rsi_full[df_rsi_full['Sector'] == sec_rsi.value]['SubIndustry']
            .dropna().unique()
        )
        sub_rsi.options = ['All'] + subs

    sub_rsi.value = 'All'


# -----------------------------
# 4. Main Analysis
# -----------------------------
def run_analysis_rsi(change):

    if getattr(run_analysis_rsi, "_running", False):
        return

    run_analysis_rsi._running = True

    try:

        sym = tick_rsi.value.upper().replace('.', '-')

        # Sync dropdowns to ticker
        if sym in df_rsi_full['Ticker'].values:

            row = df_rsi_full[df_rsi_full['Ticker'] == sym].iloc[0]

            sec_rsi.unobserve(run_analysis_rsi, 'value')
            sub_rsi.unobserve(run_analysis_rsi, 'value')

            sec_rsi.value = row['Sector']
            update_subs_rsi(None)
            sub_rsi.value = row['SubIndustry']

            sec_rsi.observe(run_analysis_rsi, 'value')
            sub_rsi.observe(run_analysis_rsi, 'value')

        df_f = df_rsi_full.copy()

        if sec_rsi.value != 'All':
            df_f = df_f[df_f['Sector'] == sec_rsi.value]

        if sub_rsi.value != 'All':
            df_f = df_f[df_f['SubIndustry'] == sub_rsi.value]

        # Clear previous figure
        out_rsi.clear_output(wait=True)

        with out_rsi:

            if df_f.empty:
                print("No data found")
                return

            df_plot = pd.concat([
                df_f.nlargest(10, 'RSI'),
                df_f.nsmallest(10, 'RSI')
            ]).drop_duplicates().sort_values('RSI')

            fig = px.bar(
                df_plot,
                y='Ticker',
                x='RSI',
                color='Ticker',
                orientation='h',
                hover_data={
                    'Sector': True,
                    'SubIndustry': True,
                    'Adj Close': ':.2f',
                    'RSI': ':.2f'
                }
            )

            fig.add_vline(x=70, line_dash="dash", line_color="red", annotation_text="Overbought")
            fig.add_vline(x=30, line_dash="dash", line_color="green", annotation_text="Oversold")

            fig.update_traces(
                texttemplate='%{y} %{x:.2f}',
                textposition='inside'
            )

            fig.update_layout(
                height=600,
                width=950,
                showlegend=False,
                title={'text': f'RSI: {sec_rsi.value} | {sub_rsi.value}', 'x':0.5},
                xaxis=dict(range=[0,100])
            )

            display(fig)

    finally:
        run_analysis_rsi._running = False


# -----------------------------
# 5. Observers
# -----------------------------
sec_rsi.observe(update_subs_rsi, 'value')

tick_rsi.observe(run_analysis_rsi, 'value')
sec_rsi.observe(run_analysis_rsi, 'value')
sub_rsi.observe(run_analysis_rsi, 'value')

# -----------------------------
# 6. Layout
# -----------------------------
display(
    widgets.VBox([
        widgets.HBox([tick_rsi, sec_rsi, sub_rsi]),
        out_rsi
    ])
)

run_analysis_rsi(None)


In [3]:
import pandas as pd
import plotly.express as px
import ta

# -----------------------------
# Function to plot RSI for last year
# -----------------------------
def plot_rsi_last_year(ticker):
    # Filter for ticker
    stock = df_long[df_long["Ticker"] == ticker].sort_values("Date").copy()
    
    # Compute RSI14 if not already present
    if "RSI14" not in stock.columns:
        stock["RSI14"] = ta.momentum.RSIIndicator(stock["Adj Close"], window=14).rsi()
    
    # Filter last year
    last_date = stock["Date"].max()
    one_year_ago = last_date - pd.Timedelta(days=365)
    stock_year = stock[stock["Date"] >= one_year_ago]
    
    # Plot line chart
    fig = px.line(
        stock_year,
        x="Date",
        y="RSI14",
        title=f"{ticker} RSI14 Over the Last Year",
        hover_data={"Adj Close": ':.2f', "RSI14": ':.2f'},
    )
    
    # Add oversold/overbought lines
    fig.add_hline(y=30, line_dash="dash", line_color="red", annotation_text="Oversold (30)", annotation_position="bottom right")
    fig.add_hline(y=70, line_dash="dash", line_color="green", annotation_text="Overbought (70)", annotation_position="top right")
    
    # Style
    fig.update_layout(
        height=600,
        width=1000,
        xaxis_title="Date",
        yaxis_title="RSI14",
        xaxis_tickfont=dict(size=14),
        yaxis_tickfont=dict(size=14),
        title=dict(x=0.5, font=dict(size=24))
    )
    
    fig.show()

# -----------------------------
# Example usage
# -----------------------------
plot_rsi_last_year("NKE")

In [4]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. UI Setup
sectors = sorted(df_long["Sector"].dropna().unique().tolist())
sector_dropdown = widgets.Dropdown(options=sectors, description="Sector:", value=sectors[0])
subindustry_dropdown = widgets.Dropdown(description="Sub-Industry:")
ticker_input = widgets.Combobox(
    description="Ticker:",
    placeholder="Type Ticker to see peers",
    ensure_option=False,
    options=sorted(df_long["Ticker"].unique().tolist())
)
out = widgets.Output()

# 2. Data Preprocessing
df_long['Date'] = pd.to_datetime(df_long['Date'])
latest_data = df_long.sort_values('Date').groupby('Ticker').last().reset_index()

def compute_ytd(df):
    ytd = df.groupby('Ticker', group_keys=False).apply(
        lambda x: (x.loc[x['Date'].dt.year == x['Date'].dt.year.max(), 'Adj Close'].iloc[-1] /
                   x.loc[x['Date'].dt.year == x['Date'].dt.year.max(), 'Adj Close'].iloc[0]) - 1,
        include_groups=False
    )
    return ytd.reset_index(name='YTD Return')

latest_data = latest_data.merge(compute_ytd(df_long), on='Ticker', how='left')

# 3. Logic: Find Ticker's Sub-Industry and filter peers
def update_table(change):
    with out:
        clear_output(wait=True)
        ticker_val = ticker_input.value.strip().upper()
        
        # Determine the filtering logic
        if ticker_val and ticker_val in latest_data['Ticker'].values:
            # Step A: Find the sub-industry for the entered ticker
            ticker_info = latest_data[latest_data['Ticker'] == ticker_val].iloc[0]
            target_sub = ticker_info['SubIndustry']
            target_sector = ticker_info['Sector']
            
            # Step B: Filter data to show all peers in that sub-industry
            df_filtered = latest_data[latest_data['SubIndustry'] == target_sub].copy()
            
            print(f"Ticker Found: {ticker_val}")
            print(f"Sector: {target_sector} | Sub-Industry: {target_sub} (Showing all peers)")
            
            # Optional: Highlight the specific ticker in the table later if needed
        else:
            # Fallback to standard dropdown filters if no valid ticker is typed
            sector = sector_dropdown.value
            sub_industry = subindustry_dropdown.value
            df_filtered = latest_data[latest_data['Sector'] == sector].copy()
            if sub_industry != 'All':
                df_filtered = df_filtered[df_filtered['SubIndustry'] == sub_industry]

        if df_filtered.empty:
            print("No data matches these criteria.")
            return
        
        # Display results
        df_display = df_filtered[['Ticker', 'Sector', 'SubIndustry', 'Adj Close', 'Return', 'YTD Return']].sort_values('Adj Close', ascending=False)
        
        # Styling: Apply a yellow background to the row matching the typed ticker
        def highlight_ticker(s):
            return ['background-color: #ffffb3' if s.Ticker == ticker_val else '' for _ in s]

        display(df_display.style.apply(highlight_ticker, axis=1).format({
            'Adj Close': "${:,.2f}",
            'Return': "{:.2%}",
            'YTD Return': "{:.2%}"
        }))

# 4. Observers
def on_sector_change(change):
    sector = change['new']
    sub_industries = sorted(latest_data[latest_data["Sector"] == sector]["SubIndustry"].dropna().unique().tolist())
    subindustry_dropdown.options = ['All'] + sub_industries
    subindustry_dropdown.value = 'All'

sector_dropdown.observe(on_sector_change, names='value')
subindustry_dropdown.observe(update_table, names='value')
ticker_input.observe(update_table, names='value')

# 5. Initialization
on_sector_change({'new': sector_dropdown.value})
display(widgets.VBox([ticker_input, sector_dropdown, subindustry_dropdown, out]))

In [5]:
import pandas as pd
import yfinance as yf
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Unique Widgets & Cache (Renamed to avoid overlap)
fund_ticker_text = widgets.Text(description='Ticker:', placeholder='e.g. MSFT', continuous_update=False)
fund_sector_drop = widgets.Dropdown(description='Sector:', options=['All'] + sorted(df_long['Sector'].unique().tolist()))
fund_sub_drop = widgets.Dropdown(description='Sub-Industry:', options=['All'])
fund_out = widgets.Output()

# Cache to prevent repeated API calls
fund_cache = {}

# 2. Update Sub-Industry options
def update_fund_subs(change):
    if fund_sector_drop.value == 'All':
        fund_sub_drop.options = ['All']
    else:
        subs = sorted(df_long[df_long['Sector'] == fund_sector_drop.value]['SubIndustry'].unique().tolist())
        fund_sub_drop.options = ['All'] + subs
    fund_sub_drop.value = 'All'

fund_sector_drop.observe(update_fund_subs, 'value')

# 3. Main Fundamentals Logic
def run_fundamentals_analysis(change):
    if getattr(run_fundamentals_analysis, '_is_syncing', False):
        return
        
    with fund_out:
        clear_output(wait=True)
        sym = fund_ticker_text.value.upper().strip().replace('.', '-')
        
        # --- SYNC DROP-DOWNS TO TICKER ---
        if sym and sym in df_long['Ticker'].values:
            row = df_long[df_long['Ticker'] == sym].iloc[-1]
            run_fundamentals_analysis._is_syncing = True
            fund_sector_drop.value = row['Sector']
            fund_sub_drop.options = ['All'] + sorted(df_long[df_long['Sector'] == row['Sector']]['SubIndustry'].unique().tolist())
            fund_sub_drop.value = row['SubIndustry']
            run_fundamentals_analysis._is_syncing = False
        
        # --- FILTER PEERS ---
        df_f = df_long.copy()
        if fund_sector_drop.value != 'All': 
            df_f = df_f[df_f['Sector'] == fund_sector_drop.value]
        if fund_sub_drop.value != 'All': 
            df_f = df_f[df_f['SubIndustry'] == fund_sub_drop.value]
            
        tickers = df_f['Ticker'].unique().tolist()
        if not tickers: 
            return print("No tickers found for this selection.")

        # --- OPTIMIZED DATA FETCHING ---
        print(f"Loading fundamentals for {len(tickers)} companies...")
        
        results = []
        for t in tickers:
            # Check cache first to save time
            if t in fund_cache:
                results.append(fund_cache[t])
                continue
                
            try:
                inf = yf.Ticker(t).info
                data_point = {
                    "Ticker": t,
                    "Net Income ($M)": inf.get('netIncomeToCommon', 0)/1e6,
                    "EBITDA ($M)": inf.get('ebitda', 0)/1e6,
                    "LFFCF ($M)": inf.get('freeCashflow', 0)/1e6,
                    "Market Cap ($M)": inf.get('marketCap', 0)/1e6,
                    "EV ($M)": inf.get('enterpriseValue', 0)/1e6,
                    "EV/EBITDA": inf.get('enterpriseToEbitda'),
                    "EPS": inf.get('trailingEps'),
                    "52W Change": inf.get('52WeekChange', 0),
                    "Beta": inf.get('beta')
                }
                fund_cache[t] = data_point 
                results.append(data_point)
            except:
                continue

        if not results:
            return print("Error fetching data.")

        df_res = pd.DataFrame(results).set_index('Ticker')
        df_res = df_res.sort_values('Market Cap ($M)', ascending=False)
        
        # Highlight the specific ticker typed in the table
        def highlight_target(s):
            return ['background-color: #ffffb3' if s.name == sym else '' for _ in s]
        
        display(df_res.style.apply(highlight_target, axis=1).format({
            'Net Income ($M)': "{:,.0f}",
            'EBITDA ($M)': "{:,.0f}",
            'LFFCF ($M)': "{:,.0f}",
            'Market Cap ($M)': "{:,.0f}",
            'EV ($M)': "{:,.0f}",
            'EPS': "{:.2f}", 
            'EV/EBITDA': "{:.2f}", 
            'Beta': "{:.2f}",
            '52W Change': "{:.2%}"
        }))

# 4. Listeners
fund_ticker_text.observe(run_fundamentals_analysis, names='value')
fund_sector_drop.observe(run_fundamentals_analysis, names='value')
fund_sub_drop.observe(run_fundamentals_analysis, names='value')

# Display
display(widgets.VBox([
    widgets.HTML("<h3>Fundamental Peer Comparison</h3>"),
    widgets.HBox([fund_ticker_text, fund_sector_drop, fund_sub_drop]), 
    fund_out
]))

In [7]:
import pandas as pd
import yfinance as yf
import ipywidgets as widgets
import plotly.express as px
from IPython.display import display, clear_output

# 1. Unique Widgets & Cache
plot_ticker_text = widgets.Text(description='Ticker:', placeholder='e.g. AAPL', continuous_update=False)
plot_sector_drop = widgets.Dropdown(description='Sector:', options=['All'] + sorted(df_long['Sector'].unique().tolist()))
plot_sub_drop = widgets.Dropdown(description='Sub-Industry:', options=['All'])
plot_out = widgets.Output()

plot_data_cache = {} 

def update_plot_subs(*args):
    if plot_sector_drop.value == 'All':
        plot_sub_drop.options = ['All']
    else:
        subs = sorted(df_long[df_long['Sector'] == plot_sector_drop.value]['SubIndustry'].unique().tolist())
        plot_sub_drop.options = ['All'] + subs
    plot_sub_drop.value = 'All'

plot_sector_drop.observe(update_plot_subs, 'value')

# 2. Visualization Logic
def run_visual_analysis(change):
    if getattr(run_visual_analysis, '_is_syncing', False):
        return
        
    with plot_out:
        clear_output(wait=True)
        sym = plot_ticker_text.value.upper().strip()
        
        if sym and sym in df_long['Ticker'].values:
            row = df_long[df_long['Ticker'] == sym].iloc[-1]
            run_visual_analysis._is_syncing = True
            plot_sector_drop.value = row['Sector']
            subs = sorted(df_long[df_long['Sector'] == row['Sector']]['SubIndustry'].unique().tolist())
            plot_sub_drop.options = ['All'] + subs
            plot_sub_drop.value = row['SubIndustry']
            run_visual_analysis._is_syncing = False

        df_f = df_long.copy()
        if plot_sector_drop.value != 'All': 
            df_f = df_f[df_f['Sector'] == plot_sector_drop.value]
        if plot_sub_drop.value != 'All': 
            df_f = df_f[df_f['SubIndustry'] == plot_sub_drop.value]
            
        tickers = sorted(df_f['Ticker'].unique().tolist())
        if not tickers: 
            return print("No tickers found for this selection.")

        print(f"Generating metrics chart for {len(tickers)} companies...")
        results = []
        for t in tickers:
            if t not in plot_data_cache:
                try:
                    inf = yf.Ticker(t).info
                    plot_data_cache[t] = {
                        "Ticker": t,
                        "Gross Margin": inf.get('grossMargins'),
                        "Op Margin": inf.get('operatingMargins'),
                        "Net Margin": inf.get('profitMargins'),
                        "P/E": inf.get('trailingPE'),
                        "Div Yield": inf.get('dividendYield'),
                        "P/B Ratio": inf.get('priceToBook'),
                        "Current Ratio": inf.get('currentRatio'),
                        "Debt/Equity": inf.get('debtToEquity'),
                        "ROA": inf.get('returnOnAssets')
                    }
                except: continue
            
            if t in plot_data_cache:
                results.append(plot_data_cache[t])

        if not results: return print("Could not fetch data.")

        df_melt = pd.DataFrame(results).melt(id_vars='Ticker', var_name='Metric', value_name='Value')

        # --- UPDATED CATEGORY ORDER ---
        fig = px.strip(df_melt, y='Ticker', x='Value', color='Ticker', 
                       facet_col='Metric', facet_col_wrap=3,
                       category_orders={
                           "Ticker": tickers,
                           "Metric": ["Gross Margin", "Op Margin", "Net Margin", # Custom order here
                                      "P/E", "Div Yield", "P/B Ratio", 
                                      "Current Ratio", "Debt/Equity", "ROA"]
                       })

        fig.update_layout(
            title={'text':'Financial Metrics Comparison','x':0.5},
            height=1000, width=1100,
            showlegend=True,
            margin=dict(l=100, t=100)
        )
        
        fig.update_traces(marker_symbol='diamond', marker_size=10)
        fig.update_xaxes(matches=None, showticklabels=True, title_text="")
        
        # Hide Y-axis labels for all but the left-most plots (1st, 4th, 7th)
        fig.for_each_yaxis(lambda y: y.update(showticklabels=False, title_text="") 
                           if y.anchor not in ['x', 'x4', 'x7'] 
                           else y.update(title_text=""))

        fig.update_yaxes(matches='y')

        for annot in fig.layout.annotations:
            annot.text = annot.text.split("=")[-1] 
        
        fig.show()

# 3. Listeners
plot_ticker_text.observe(run_visual_analysis, names='value')
plot_sector_drop.observe(run_visual_analysis, names='value')
plot_sub_drop.observe(run_visual_analysis, names='value')

display(widgets.VBox([
    widgets.HTML("<h4>Visual Peer Benchmarking</h4>"),
    widgets.HBox([plot_ticker_text, plot_sector_drop, plot_sub_drop]), 
    plot_out
]))